# 01. 인프라 확인 + GR00T Base 모델 검증

사전 배포된 GR00T용 인프라(ECR/CodeBuild/SageMaker/MLflow)를 확인하고, fine-tune 전에 base 모델이 GR00T Policy Server에서 정상 추론되는지 스모크 테스트합니다. 워크숍 가이드 **모듈 3**과 함께 진행합니다.

**선행 조건**

- `setup-notebooks.sh` 실행 완료 (모듈 3.3) — `config.yaml` 채우기 + 커널 `GR00T (uv)` 등록
- GR00T Policy Server 실행 중 (모듈 3.6)

## 1단계: config 확인

`setup-notebooks.sh`가 채운 `config.yaml`에서 계정/리전/리소스 정보를 읽습니다.

In [ ]:
from pathlib import Path
import yaml

CONFIG = yaml.safe_load((Path.cwd().parent / "config.yaml").read_text())
aws = CONFIG["aws"]
ACCOUNT_ID, REGION = aws["account_id"], aws.get("region", "us-east-1")
assert ACCOUNT_ID and aws["bucket_name"], "config.yaml 값 누락 — setup-notebooks.sh를 먼저 실행하세요 (모듈 3.3)"
aws, CONFIG.get("ecr", {})

## 2단계: 사전 배포된 GrootFinetune 스택 확인

워크숍 환경에서는 `GrootFinetune-<ACCOUNT_ID>` 스택이 프로비저닝 시 이미 배포되어 있습니다. CloudFormation Outputs를 조회해 리소스가 준비됐는지 확인합니다.

In [ ]:
import boto3

cfn = boto3.client("cloudformation", region_name=REGION)
outputs = cfn.describe_stacks(StackName=f"GrootFinetune-{ACCOUNT_ID}")["Stacks"][0]["Outputs"]
for o in sorted(outputs, key=lambda x: x["OutputKey"]):
    print(f"{o['OutputKey']:34s} {o['OutputValue']}")

### (선택) 수동 배포

자가진행(self-paced) 등 스택이 아직 없는 환경에서만 사용합니다. **워크숍 환경에서는 실행하지 마세요** — 이미 배포되어 있고, 배포에는 10분 이상 걸립니다.

In [ ]:
# 스택이 없는 자가진행 환경에서만 주석을 해제해 실행하세요 (10분 이상 소요).
# !cd ../../infra/groot && npm ci --silent && npm run deploy -- -c region={REGION}
# !cd ../../infra/groot && node_modules/.bin/ts-node bin/update-config.ts --region {REGION}

## 3단계: GR00T Policy Server 검증 (ZMQ)

Policy Server는 모듈 3.6에서 띄운 상태여야 합니다 (`docker ps --filter name=groot-policy-server`로 확인). REQ 소켓으로 `ping`을 보내 응답을 확인합니다.

In [ ]:
import zmq, msgpack

SERVER_IP = "127.0.0.1"   # 원격이면 인스턴스 IP
ctx = zmq.Context(); sock = ctx.socket(zmq.REQ); sock.setsockopt(zmq.RCVTIMEO, 10000)
sock.connect(f"tcp://{SERVER_IP}:5555")
sock.send(msgpack.packb({"endpoint": "ping"}))
print("ping 응답:", msgpack.unpackb(sock.recv()))

## 4단계: 더미 observation으로 get_action 호출 (GR1 임베디먼트)

`inference/batch-zmq/test_inference_remote.py`와 동일한 방식으로 numpy 배열을 `.npy` 바이트로 감싸 msgpack으로 인코딩/디코딩합니다.

In [ ]:
import numpy as np, io

def encode_ndarray(obj):
    if isinstance(obj, np.ndarray):
        buf = io.BytesIO()
        np.save(buf, obj, allow_pickle=False)
        return {"__ndarray_class__": True, "as_npy": buf.getvalue()}
    return obj

def decode_ndarray(obj):
    if isinstance(obj, dict) and "__ndarray_class__" in obj:
        return np.load(io.BytesIO(obj["as_npy"]), allow_pickle=False)
    return obj

In [ ]:
# 더미 observation (GR1 임베디먼트)
observation = {
    "video": {
        "ego_view_bg_crop_pad_res256_freq20": np.random.randint(0, 255, (1, 1, 256, 256, 3), dtype=np.uint8)
    },
    "state": {
        "left_arm": np.random.rand(1, 1, 7).astype(np.float32),
        "right_arm": np.random.rand(1, 1, 7).astype(np.float32),
        "left_hand": np.random.rand(1, 1, 6).astype(np.float32),
        "right_hand": np.random.rand(1, 1, 6).astype(np.float32),
        "waist": np.random.rand(1, 1, 3).astype(np.float32),
    },
    "language": {"task": [["pick up the cup"]]}
}

request = {"endpoint": "get_action", "data": {"observation": observation}}
sock.send(msgpack.packb(request, default=encode_ndarray))
response = msgpack.unpackb(sock.recv(), raw=False, object_hook=decode_ndarray)

if isinstance(response, list):
    action = response[0]
    print("추론 성공! Action keys:", list(action.keys()))
    for key in action:
        print(f"  {key}: shape={np.array(action[key]).shape}")
elif isinstance(response, dict) and "error" in response:
    print("에러:", response["error"])
else:
    print("예상치 못한 응답:", type(response))

## 정리

base 모델이 응답하면 인프라·서빙이 정상입니다. 다음은 모듈 4(`02_sagemaker_pipeline.ipynb`) — fine-tuning 파이프라인 실행입니다. fine-tune 결과의 closed-loop 검증은 모듈 5(`03_closed_loop_eval.ipynb`)에서 진행합니다.